In [1]:
import os, ast, random
import numpy as np
import pandas as pd
from PIL import Image
import hashlib
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from tqdm import tqdm
import networkx as nx
from sklearn.model_selection import StratifiedGroupKFold

SEED = 42
random.seed(SEED); np.random.seed(SEED)

DATA_DIR = '/kaggle/input/datasets/uriyoung11/snuaichallenge/snuaichallenge_data'
N_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3  # 이 값 초과로 등장하는 해시는 "우연/재사용 이미지"로 간주, 클러스터링에서 제외

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
train['answer_tuple'] = train['Answer'].apply(lambda x: str(ast.literal_eval(x)))

In [2]:
def compute_md5(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def get_all_image_records(df, img_dir):
    records = []
    for _, row in df.iterrows():
        for i in range(1, 5):
            path = f"{img_dir}/{row['Id']}/{row[f'Input_{i}']}"
            records.append({'sample_id': row['Id'], 'input_slot': f'Input_{i}', 'path': path})
    return records

print("Train 이미지 해시 계산 중...")
train_records = get_all_image_records(train, f'{DATA_DIR}/train')
paths = [r['path'] for r in train_records]
with ThreadPoolExecutor(max_workers=32) as executor:
    hashes = list(tqdm(executor.map(compute_md5, paths), total=len(paths)))
for r, h in zip(train_records, hashes):
    r['hash'] = h

Train 이미지 해시 계산 중...


100%|██████████| 38140/38140 [00:09<00:00, 3870.24it/s]


In [3]:
hash_counter = defaultdict(set)
for r in train_records:
    hash_counter[r['hash']].add(r['sample_id'])

# MAX_SAMPLES_PER_HASH 초과 해시는 "우연 재사용"으로 간주, 그래프 연결에서 제외
hash_to_samples_final = {h: s for h, s in hash_counter.items() if len(s) <= MAX_SAMPLES_PER_HASH}
excluded = sum(1 for h, s in hash_counter.items() if len(s) > MAX_SAMPLES_PER_HASH)
print(f"제외된 해시(빈번 재사용) 수: {excluded}")

G = nx.Graph()
G.add_nodes_from(train['Id'].unique())
for h, sample_ids in hash_to_samples_final.items():
    sample_ids = list(sample_ids)
    for i in range(len(sample_ids)):
        for j in range(i+1, len(sample_ids)):
            G.add_edge(sample_ids[i], sample_ids[j])

components = list(nx.connected_components(G))
cluster_sizes = pd.Series([len(c) for c in components])
print(f"총 {len(components)}개 클러스터")
print(f"상위 10개 크기:\n{cluster_sizes.sort_values(ascending=False).head(10)}")

id_to_cluster = {}
for cluster_id, comp in enumerate(components):
    for sid in comp:
        id_to_cluster[sid] = cluster_id
train['cluster_id'] = train['Id'].map(id_to_cluster)

제외된 해시(빈번 재사용) 수: 2081
총 8147개 클러스터
상위 10개 크기:
156    14
112     9
121     9
629     7
308     7
80      7
577     7
409     7
119     7
694     7
dtype: int64


In [4]:
def format_training_sample(row, img_dir):
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']
    answer = ast.literal_eval(row['Answer'])  # 예: [3,4,1,2] (Input_i가 몇 번째인지)

    # 대회 정답 형식(Input_i의 시간위치)을 "시간순 프레임 번호 나열"로 변환
    # 예: answer=[3,4,1,2] -> 1번째=Input_3, 2번째=Input_4, 3번째=Input_1, 4번째=Input_2
    #     -> 모델이 생성할 타겟 텍스트: [3,4,1,2]  (Input 번호를 시간순으로 나열)
    temporal_order = [0, 0, 0, 0]
    for input_idx, pos in enumerate(answer):  # input_idx: 0~3 (Input_1~4), pos: 1~4
        temporal_order[pos - 1] = input_idx + 1
    target_text = str(temporal_order)  # 베이스라인 프롬프트가 기대하는 출력 형식과 동일

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(img_dir, row['Id'], img_file)
        content.append({"type": "image", "image": img_path})
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})

    messages = [
        {"role": "user", "content": content},
        {"role": "assistant", "content": [{"type": "text", "text": target_text}]}
    ]
    return messages, target_text

In [5]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train['fold'] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(train, train['answer_tuple'], groups=train['cluster_id'])):
    train.loc[val_idx, 'fold'] = fold

# 검증 1: 같은 클러스터가 여러 fold에 걸쳐있지 않은지 (0이어야 정상)
cluster_fold_counts = train.groupby('cluster_id')['fold'].nunique()
print(f"여러 fold에 걸친 클러스터 수: {(cluster_fold_counts > 1).sum()} (0이어야 정상)")

# 검증 2: fold별 샘플 수 균형
print(f"\nFold별 샘플 수:\n{train['fold'].value_counts().sort_index()}")

# 검증 3: fold별 순열 클래스 비율 균형 (참고용)
fold_balance = train.groupby('fold')['answer_tuple'].value_counts(normalize=True).unstack().std()
print(f"\nFold간 순열 클래스 비율 표준편차: {fold_balance.mean():.5f}")

여러 fold에 걸친 클러스터 수: 0 (0이어야 정상)

Fold별 샘플 수:
fold
0    1915
1    1905
2    1886
3    1911
4    1918
Name: count, dtype: int64

Fold간 순열 클래스 비율 표준편차: 0.00403


In [6]:
import imagesize

def check_portrait_fast(row_dict):
    for i in range(1, 5):
        path = f"{DATA_DIR}/train/{row_dict['Id']}/{row_dict[f'Input_{i}']}"
        w, h = imagesize.get(path)
        if h > w:
            return True
    return False

rows = train.to_dict('records')
with ThreadPoolExecutor(max_workers=32) as executor:
    results = list(tqdm(executor.map(check_portrait_fast, rows), total=len(rows)))
train['is_portrait'] = results
print(f"세로형 샘플 수: {train['is_portrait'].sum()} / {len(train)}")

100%|██████████| 9535/9535 [00:12<00:00, 749.90it/s]

세로형 샘플 수: 297 / 9535


In [7]:
%pip install -q peft bitsandbytes qwen-vl-utils

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from qwen_vl_utils import process_vision_info
from scipy.stats import kendalltau

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "Qwen/Qwen2-VL-2B-Instruct"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 48.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [8]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    min_pixels=128*28*28,   # 하한
    max_pixels=256*28*28,   # 상한 - 이 값을 낮출수록 토큰 수 감소, 속도 개선
)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

print(f"GPU 메모리: {torch.cuda.memory_allocated()/1e9:.1f}GB")

preprocessor_config.json:   0%|          | 0.00/347 [00:00<?, ?B/s]

chat_template.json: 0.00B [00:00, ?B/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/272 [00:00<?, ?B/s]

trainable params: 4,358,144 || all params: 2,213,343,744 || trainable%: 0.1969
GPU 메모리: 0.2GB


In [9]:
def format_training_sample(row, img_dir):
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']
    answer = ast.literal_eval(row['Answer'])  # Input_i가 몇 번째인지

    # "시간순 프레임 번호 나열" 형식으로 변환 (베이스라인 파싱과 호환)
    temporal_order = [0, 0, 0, 0]
    for input_idx, pos in enumerate(answer):
        temporal_order[pos - 1] = input_idx + 1
    target_text = str(temporal_order)

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(img_dir, row['Id'], img_file)
        content.append({"type": "image", "image": img_path})
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})

    messages = [
        {"role": "user", "content": content},
        {"role": "assistant", "content": [{"type": "text", "text": target_text}]}
    ]
    return messages, target_text


def parse_model_output(output_text):
    try:
        start_idx = output_text.find('[')
        end_idx = output_text.rfind(']')
        if start_idx != -1 and end_idx != -1:
            result = ast.literal_eval(output_text[start_idx:end_idx+1])
            if isinstance(result, list) and len(result) == 4 and sorted(result) == [1,2,3,4]:
                submission_answer = [0]*4
                for index, image_num in enumerate(result):
                    submission_answer[image_num - 1] = index + 1
                return submission_answer, True
    except:
        pass
    return [1,2,3,4], False

In [10]:
class QwenOrderingDataset(Dataset):
    def __init__(self, df, img_dir, processor):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        messages, target_text = format_training_sample(row, self.img_dir)

        full_text = self.processor.apply_chat_template(messages, tokenize=False)
        prompt_only = self.processor.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)

        image_inputs, _ = process_vision_info(messages)

        full_enc = self.processor(text=[full_text], images=image_inputs, return_tensors="pt", padding=False)
        prompt_enc = self.processor(text=[prompt_only], images=image_inputs, return_tensors="pt", padding=False)

        input_ids = full_enc['input_ids'][0]
        labels = input_ids.clone()
        prompt_len = prompt_enc['input_ids'].shape[1]
        labels[:prompt_len] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': full_enc['attention_mask'][0],
            'labels': labels,
            'pixel_values': full_enc['pixel_values'],
            'image_grid_thw': full_enc['image_grid_thw'],
        }


def collate_fn(batch):
    max_len = max(item['input_ids'].shape[0] for item in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids, attn_mask, labels = [], [], []
    for item in batch:
        seq_len = item['input_ids'].shape[0]
        pad_len = max_len - seq_len
        input_ids.append(torch.cat([item['input_ids'], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        attn_mask.append(torch.cat([item['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))

    pixel_values = torch.cat([item['pixel_values'] for item in batch], dim=0)
    image_grid_thw = torch.cat([item['image_grid_thw'] for item in batch], dim=0)

    return {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attn_mask),
        'labels': torch.stack(labels),
        'pixel_values': pixel_values,
        'image_grid_thw': image_grid_thw,
    }

In [11]:
fold = 0
tr_df = train[train['fold'] != fold].reset_index(drop=True)
va_df = train[train['fold'] == fold].reset_index(drop=True)
print(f"Train: {len(tr_df)}, Val: {len(va_df)}")

tr_ds = QwenOrderingDataset(tr_df, f'{DATA_DIR}/train', processor)
tr_loader = DataLoader(tr_ds, batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=4, pin_memory=True)

print(f"총 스텝 수 (1 epoch): {len(tr_loader)}")
print(f"예상 소요 시간: 약 {len(tr_loader) * 5.9 / 3600:.1f}시간")

from collections import deque

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)
GRAD_ACCUM_STEPS = 8
EPOCHS = 1
CHECKPOINT_EVERY = 1000  # 중간 저장 주기 (세션 끊김 대비)

model.train()
loss_window = deque(maxlen=100)

for epoch in range(EPOCHS):
    optimizer.zero_grad()
    total_loss = 0
    step_count = 0

    for step, batch in enumerate(tqdm(tr_loader, desc=f"Epoch {epoch}")):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        step_loss = loss.item() * GRAD_ACCUM_STEPS
        loss_window.append(step_loss)
        total_loss += step_loss
        step_count += 1

        if step % 100 == 0:
            print(f" step {step}/{len(tr_loader)}: loss={step_loss:.4f}, MA100={np.mean(loss_window):.4f}")

        # 중간 체크포인트 (세션 끊김/타임아웃 대비)
        if (step + 1) % CHECKPOINT_EVERY == 0:
            model.save_pretrained(f'/kaggle/working/qwen_lora_checkpoint_step{step+1}')
            print(f" [체크포인트 저장] step {step+1}")

    print(f"Epoch {epoch} 평균 loss: {total_loss/step_count:.4f}")

model.save_pretrained('/kaggle/working/qwen_lora_full_epoch1')
print("최종 모델 저장 완료")

Train: 7620, Val: 1915
총 스텝 수 (1 epoch): 7620
예상 소요 시간: 약 12.5시간


Epoch 0:   0%|          | 0/7620 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: The AccumulateGrad node's stream does not match the stream of the node that produced the incoming gradient. This may incur unnecessary synchronization and break CUDA graph capture if the AccumulateGrad node's stream is the default stream. This mismatch is caused by an AccumulateGrad nod

 step 0/7620: loss=0.3828, MA100=0.3828


Epoch 0:   1%|▏         | 101/7620 [08:42<11:15:49,  5.39s/it]

 step 100/7620: loss=0.2693, MA100=0.2248


...

 step 7500/7620: loss=0.2307, MA100=0.1264


Epoch 0: 100%|█████████▉| 7601/7620 [10:50:34<01:35,  5.02s/it]

 step 7600/7620: loss=0.0710, MA100=0.1329


Epoch 0: 100%|██████████| 7620/7620 [10:52:17<00:00,  5.14s/it]


Epoch 0 평균 loss: 0.1556
최종 모델 저장 완료


In [12]:
def compute_metrics(pred, target):
    exact_match = (pred == target).all(axis=1).mean()
    position_acc = (pred == target).mean()
    taus = [kendalltau(p,t)[0] if not np.isnan(kendalltau(p,t)[0]) else 1.0 for p,t in zip(pred,target)]
    return {'exact_match': exact_match, 'position_acc': position_acc, 'kendall_tau': np.mean(taus)}

model.eval()
va_df_sample = va_df.sample(n=min(150, len(va_df)), random_state=SEED).reset_index(drop=True)

results = []
for idx, row in tqdm(va_df_sample.iterrows(), total=len(va_df_sample)):
    messages, _ = format_training_sample(row, f'{DATA_DIR}/train')
    text = processor.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(messages[:1])
    inputs = processor(text=[text], images=image_inputs, padding=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_text = processor.batch_decode(trimmed, skip_special_tokens=True)[0]

    pred, parse_ok = parse_model_output(output_text)
    results.append({'Id': row['Id'], 'pred': pred, 'parse_ok': parse_ok})

pred_array = np.array([r['pred'] for r in results]) - 1
target_array = np.array([ast.literal_eval(a) for a in va_df_sample['Answer']]) - 1

overall = compute_metrics(pred_array, target_array)
no_order_mask = va_df_sample['No_ordering'].values
shuffled = compute_metrics(pred_array[~no_order_mask], target_array[~no_order_mask])

print(f"[LoRA 파인튜닝 후, Group-CV val, n={len(va_df_sample)}]")
print(f"  전체 exact={overall['exact_match']:.4f}")
print(f"  셔플됨 exact={shuffled['exact_match']:.4f}")
print(f"  파싱 실패율={1-np.mean([r['parse_ok'] for r in results]):.1%}")

pred_strs = [str(r['pred']) for r in results]
from collections import Counter
print("\n예측 분포 상위 5개:", Counter(pred_strs).most_common(5))

100%|██████████| 150/150 [06:22<00:00,  2.55s/it]

[LoRA 파인튜닝 후, Group-CV val, n=150]
  전체 exact=0.3800
  셔플됨 exact=0.2698
  파싱 실패율=0.0%

예측 분포 상위 5개: [('[1, 2, 3, 4]', 64), ('[2, 3, 4, 1]', 8), ('[4, 1, 2, 3]', 8), ('[3, 2, 4, 1]', 7), ('[1, 3, 4, 2]', 7)]
